### Imports and General Functions

In [1]:
from itertools import chain, combinations, product
import itertools
import numpy as np

#  Descr: get domain of the generators
#  Input: specific generator list (list with lists)
# Output: list with all symptoms (domain)
def getdom(gens):
    all_lists = []

    def recurse(nested):
        for element in nested:
            if isinstance(element, set):
                all_lists.append(list(element))
            elif isinstance(element, list):
                recurse(element)
    recurse(gens)
    
    return [item for sublist in all_lists for item in sublist]

### Generator Definitions

In [2]:
# Note: These definitions differ slightly in ordering, but are equivalent in content.

# cond_gen0: [{...}, k]
# cond_gen1: [{...}, {...}, ..., k]
# cond_gen2: [[{.},...], [{.},...], (l,m,n)]
# cond_gen3: [[{.},...], [{.},...]]

# Generator 0
#############
#  Descr: generate the elements of the powerset with at least k elements
#         (alternative function name: cond_gen0)
#  Input: list or set of symptoms, condition k
# Output: itertools.chain (needs to be iterated over and transformed to list with sets)
def powerset(s, k=0):
    s = list(s)
    return chain.from_iterable(combinations(s,r) for r in range(k,len(s)+1))

# Generator 1
#############
#  Descr: generate a list with sets with at least k elements from k different sets
#  Input: list with sets e.g. [{...},{...},...,{...}], condition k      
# Output: list with sets
def cond_gen1(sets, k):
    # generate the powerset
    pset = []
    pset = [{*s} for s in powerset(set.union(*sets))]
    
    # check if the condition is met for each element
    fin = []
    for ps in pset:
        c = len([1 for s in sets if len(s&ps)])
        if c >= k:
            fin.append(ps)
    return fin

# Generator 2
#############
#  Descr: generate a list with sets with at least k elements from first, l elements from second and m elements from both lists
#  Input: list with 2 lists containing sets e.g. [[{.},...],[{.},...]], conditions k in the form (l,m,n)
# Output: list with sets
def cond_gen2(lists, k):
    # generate the powersets for each list
    # make the product of each element in each set and unite them
    pset0, pset1, prod_pset, prod_uset = [], [], [], []
    pset0 = [{*s} for s in powerset(set.union(*lists[0]))]
    pset1 = [{*s} for s in powerset(set.union(*lists[1]))]
    prod_pset = [set.union(*p) for p in itertools.product(pset0, pset1)]
    
    # check if the conditions are met for each element
    fin = []
    for ps in prod_pset:
        c0 = len([1 for s in lists[0] if len(s&ps)])
        c1 = len([1 for s in lists[1] if len(s&ps)])
        
        if c0>=k[0] and c1>=k[1] and c0+c1>=k[2]:
            fin.append(ps)
    return fin

# Generator 3
#############
#  Descr: just cartesian product of 2 lists
#  Input: list with 2 lists containing sets e.g. [[{.},...],[{.},...]], conditions k in the form (l,m,-n)
# Output: list with sets
def cond_gen3(lists):
    return [set.union(*p) for p in itertools.product(lists[0], lists[1])]

### Combine Generators

In [3]:
#  Descr: yield results from itertools product function
#  Input: list of generated lists
# Output: cartesian product of all elements (sets) between all lists
def gen_comb_yield(*list):
    for combi in itertools.product(*list):
        yield combi

#  Descr: expands all the generators into lists, combines them (cartesion product) and prints them
#  Input: list of the generators, return list, rate of return - every Xs element
# Output: - (cartesian products per print)
def gen_all_comb(list_of_gens, ret=False, ret_rate=0):
    list_of_combis = []
    for gen in list_of_gens:
        # first generator element is a list -> #2/#3
        if isinstance(gen[0], list):
            # generator len is over 2 -> #2
            if len(gen)>2:
                generated = cond_gen2(gen[:-1], gen[-1])
                list_of_combis.append(generated)
            elif len(gen)==2:
                generated = cond_gen3(gen)
                list_of_combis.append(generated) 
            else:
                list_of_combis.append(gen[0])
        # first generator element is a set -> #0/#1        
        else:
            # generator len is over 2 -> #1
            if len(gen)>2:
                generated = cond_gen1(gen[:-1], gen[-1])
                list_of_combis.append(generated)
            elif len(gen)==2:
                generated = [{*x} for x in powerset(gen[0], gen[1])]
                list_of_combis.append(generated)
            else:
                list_of_combis.append(gen)
    
    #domain = getdom(list_of_gens)
    combi_generator = gen_comb_yield(*list_of_combis)    
    
    counter = 0
    if ret:
        return_list = []
        
        #return every ret_rate element
        if ret_rate > 0: 
            for c in combi_generator:
                if counter % ret_rate == 0:
                    return_list.append(set.union(*c))
                counter = counter + 1
                
        #return all
        else:
            for c in combi_generator:
                return_list.append(set.union(*c))
                counter = counter + 1
        
        print()
        print(f'{counter:,}',"symptom combinations generated!")
                
        return return_list
    else:
        for c in combi_generator:
            # comment next line to only see the count
            #print(set.union(*c))
            counter = counter + 1
        print()
        print(f'{counter:,}',"symptom combinations generated!")

### Example Generators

In [4]:
# Generator 1 - [{...}, k]
gens1 = [
    [{'a1','a2','a3'}, 2]
]
# Generator 2 - [{...}, {...}, ..., k]
gens2 = [
    [{'b1','b2'},{'c1','c2','c3'},{'d1','d2'}, 2]
]
# Generator 3 - [[{.},...], [{.},...], (l,m,n)]
gens3 = [
    [[{'e1'},{'f1','f2'}], [{'g1'},{'h1','h2'},{'i1','i2'}], (1,0,3)]
]
# Generator 4 - [[{.},...], [{.},...]]
gens4 = [
    [[{'j1'},{'j2'}], [{'k1'},{'k2'}]]
]
# Generators 1-4 combined
gens5 = [
    [{'a1','a2','a3'}, 2],
    [{'b1','b2'},{'c1','c2','c3'},{'d1','d2'}, 2],
    [[{'e1'},{'f1','f2'}], [{'g1'},{'h1','h2'},{'i1','i2'}], (1,0,3)],
    [[{'j1'},{'j2'}], [{'k1'},{'k2'}]]
]

### Major Depressive Disorder

In [5]:
# Major Depressive Disorder - A
gens_mdd_A = [
   [[{'depressed_mood'}, {'loss_of_interest', 'loss_of_pleasure'}],
    [{'increased_appetite', 'reduced_appetite', 'weight_gain', 'weight_loss'}, 
     {'hypersomnia', 'insomnia'},
     {'psychomotor_agitation', 'psychomotor_retardation'},
     {'loss_of_energy', 'fatigue'},
     {'feeling_worthless', 'feeling_excessive_guilt', 'feeling_inappropriate_guilt'},
     {'diminished_ability_to_concentrate', 'diminished_ability_to_think', 'indecisiveness'},
     {'recurrent_death_thoughts', 'recurrent_suicidal_ideation', 'plan_for_committing_suicide', 'suicide_attempt'}],(1,0,5)]  
] #7,283,511 CSSCs

# Major Depressive Disorder - AB
gens_mdd_AB = [
   [[{'depressed_mood'}, {'loss_of_interest', 'loss_of_pleasure'}],
    [{'increased_appetite', 'reduced_appetite', 'weight_gain', 'weight_loss'}, 
     {'hypersomnia', 'insomnia'},
     {'psychomotor_agitation', 'psychomotor_retardation'},
     {'loss_of_energy', 'fatigue'},
     {'feeling_worthless', 'feeling_excessive_guilt', 'feeling_inappropriate_guilt'},
     {'diminished_ability_to_concentrate', 'diminished_ability_to_think', 'indecisiveness'},
     {'recurrent_death_thoughts', 'recurrent_suicidal_ideation', 'plan_for_committing_suicide', 'suicide_attempt'}],(1,0,5)],
   [{'significant_distress_social_area', 'significant_distress_occupational_area', 
     'significant_distress_other_important_area', 'significant_impairment_social_area',
     'significant_impairment_occupational_area', 'significant_impairment_other_important_area'}, 1]
] #458,861,193 CSSCs

# Major Depressive Disorder - CDE
gens_mdd_CDE = [
    [{'not_attributable_to_the_physiological_effects_of_a_substance',
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}],
    [{'not_better_explained_by_schizoaffective_disorder',
      'not_better_explained_by_schizophrenia',
      'not_better_explained_by_schizophreniform_disorder',
      'not_better_explained_by_delusional_disorder',
      'not_better_explained_by_other_specified_schizophrenia_spectrum',
      'not_better_explained_by_other_unspecified_schizophrenia_spectrum',
      'not_better_explained_by_other_psychotic_disorder'}],
    [{'no_manic_episode', 'no_hypomanic_episode'}, 1]
]

# Major Depressive Disorder Dummy
gens_mdd_dummy = [
   [[{'a1'}, {'b1', 'b2'}],
    [{'c1','c2'}, 
     {'d1','d2'},
     {'e1','e2'},
     {'f1','f2'},
     {'g1','g2'},
     {'l1','l2'},
     {'m1','m2'}], (1,0,5)],
   [{'h1','h2','h3','h4','h5','h6'}, 1],
   [{'i1','i2'}],
   [{'j1','j2','j3','j4','j5'}],
   [{'k1','k2'}, 1]
]

gens_major_depressive = [
   [[{'depressed_mood'}, {'loss_of_interest', 'loss_of_pleasure'}],
    [{'increased_appetite', 'reduced_appetite', 'weight_gain', 'weight_loss'}, 
     {'hypersomnia', 'insomnia'},
     {'psychomotor_agitation', 'psychomotor_retardation'},
     {'loss_of_energy', 'fatigue'},
     {'feeling_worthless', 'feeling_excessive_guilt', 'feeling_inappropriate_guilt'},
     {'diminished_ability_to_concentrate', 'diminished_ability_to_think', 'indecisiveness'},
     {'recurrent_death_thoughts', 'recurrent_suicidal_ideation', 
      'plan_for_committing_suicide', 'suicide_attempt'}], (1,0,5)],
   [{'significant_distress_social_area', 'significant_distress_occupational_area', 
     'significant_distress_other_important_area', 'significant_impairment_social_area',
     'significant_impairment_occupational_area', 'significant_impairment_other_important_area'}, 1],
   [{'not_attributable_to_the_physiological_effects_of_a_substance',
     'not_attributable_to_the_physiological_effects_of_another_medical_condition'}],
   [{'not_better_explained_by_schizoaffective_disorder',
     'not_better_explained_by_schizophrenia',
     'not_better_explained_by_schizophreniform_disorder',
     'not_better_explained_by_delusional_disorder',
     'not_better_explained_by_other_specified_schizophrenia_spectrum',
     'not_better_explained_by_other_unspecified_schizophrenia_spectrum',
     'not_better_explained_by_other_psychotic_disorder'}],
   [{'no_manic_episode', 'no_hypomanic_episode'}, 1]
] #1,376,583,579 CSSCs

### Other Disorders

In [6]:
gens_generalized_anxiety = [
    [{'anxiety', 'worry', 'difficulty_to_control_the_worry'}], #A+B
    [{'restlessness','feeling_keyed_up','feeling_on_edge'},
     {'fatigue'},
     {'difficulty_concentrating','mind_going_blank'}, 
     {'irritability'},
     {'muscle_tension'},
     {'sleep_disturbance'}, 3], #C
    [{'significant_distress_social_area', 'significant_distress_occupational_area', 
      'significant_distress_other_important_area', 'significant_impairment_social_area', 
      'significant_impairment_occupational_area', 'significant_impairment_other_important_area'}, 1], #D
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}], #E
    [{'not_better_explained_by_panic_disorder',
      'not_better_explained_by_social_anxiety_disorder',
      'not_better_explained_by_obsessive_compulsive_disorder',
      'not_better_explained_by_separation_anxiety_disorder',
      'not_better_explained_by_posttraumatic_stress_disorder',
      'not_better_explained_by_anorexia_nervosa',
      'not_better_explained_by_somatic_symptom_disorder',
      'not_better_explained_by_body_dysmorphic_disorder',
      'not_better_explained_by_illness_anxiety_disorder',
      'not_better_explained_by_schizophrenia',
      'not_better_explained_by_delusional_disorder'}]  #F
] #27,090 CSSCs

gens_schizophrenia = [
    [[{'delusions'}, {'hallucinations'}, {'disorganized_speech'}],
     [{'grossly_disorganized_behavior', 'catatonic_behavior'}, {'negative_symptoms'}], (1,0,2)], #A
    [{'decreased_level_of_functioning_work'},
     {'decreased_level_of_functioning_interpersonal_relations'},
     {'decreased_level_of_functioning_self_care'}, 1], #B
    [{'duration_at_least_6_months'}], #C
    [{'not_better_explained_by_schizoaffective_disorder',
      'not_better_explained_by_bipolar_disorder_with_psychotic_features',
      'not_better_explained_by_autism_spectrum_disorder'}], #D
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}], #E
] #371 CSSCs

gens_schizoaffective = [
    [[{'delusions'}, {'hallucinations'}, {'disorganized_speech'}],
     [{'depressive_episode'}, {'manic_episode'}], (1,0,2)], #A
    [{'delusions_for_at_least_2_weeks_without_mood_episode'},
     {'hallucinations_for_at_least_2_weeks_without_mood_episode'}, 1], #B
    [{'mood_symptoms_present_majority_of_total_duration'}], #C
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}] #D
] #75 CSSCs

gens_schizophreniform = [
    [[{'delusions'}, {'hallucinations'}, {'disorganized_speech'}],
     [{'grossly_disorganized_behavior', 'catatonic_behavior'}, {'negative_symptoms'}], (1,0,2)], #A
    [{'duration_at_least_1_month', 'duration_less_than_6_months'}], #B
    [{'not_better_explained_by_schizoaffective_disorder',
      'not_better_explained_by_bipolar_disorder_with_psychotic_features'}], #C
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}] #D
] #53 CSSCs

gens_delusional = [
    [{'delustions', 'duration_at_least_1_month'}], #A
     #B,C,D are conditional or something like "this is not present"
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition',
      'not_better_explained_by_body_dysmorphic_disorder',
      'not_better_explained_by_obsessive_compulsive_disorder'}], #E
] #1 CSSC

gens_speech_sound = [
    [{'persistent_difficulty_with_speech_sound_production', 
      'interference_with_speech_intelligibility', 
      'prevents_verbal_communication_of_messages'}], #A
    [{'limitations_in_effective_communication_social_participation',
      'limitations_in_effective_communication_academic_achievement',
      'limitations_in_effective_communication_occupational_performance'}, 1], #B
    [{'onset_early_developmental_period'}], #C
    [{'not_attributable_to_cerebral_palsy', 
      'not_attributable_to_cleft_palate',
      'not_attributable_to_deafness', 'not_attributable_to_hearing_loss',
      'not_attributable_to_traumatic_brain_injury',
      'not_attributable_to_other_medical_condition'}], #D
] #7 CSSCs

gens_persistent_depressive = [
    [{'depressed_mood'}], #A
    [{'poor_appetite', 'overeating'}, 
     {'insomnia', 'hypersomnia'},
     {'fatigue', 'low_energy'},
     {'low_self_esteem'},
     {'poor_concentration', 'indecisiveness'},
     {'hopelessness'}, 2], #B
    [{'duration_at_least_2_years'}], #C
    #D is useless as it says "may be present"
    [{'no_past_manic_episodes', 'no_past_hypomanic_episode', 'no_past_cyclothymic_disorder'}], #E
    [{'not_better_explained_by_schizoaffective_disorder',
      'not_better_explained_by_schizophrenia',
      'not_better_explained_by_delusional_disorder'}], #F
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}], #G
    [{'significant_distress_social_area', 'significant_distress_occupational_area', 
      'significant_distress_other_important_area', 'significant_impairment_social_area', 
      'significant_impairment_occupational_area', 'significant_impairment_other_important_area'}, 1], #H
] #63,567 CSSCs

gens_panic = [
    [{'intense_fear','intense_discomfort'}, 1], 
    [{'palpitations','pounding_heart','accelerated_heart_rate'},
     {'sweating'},
     {'trembling','shaking'},
     {'shortness_of_breath','smothering'},
     {'feelings_of_choking'},
     {'chest_pain','discomfort'},
     {'nausea','abdominal_distress'},
     {'feeling_dizzy','feeling_unsteady','feeling_light_headed','feeling_faint'},
     {'chills','heat_sensations'},
     {'paresthesias'},
     {'derealization','depersonalization'},
     {'fear_of_losing_control','fear_of_going_crazy'},
     {'fear_of_dying'}, 4], #A
    [{'persistent_concern_about_additional_panic_attack','persistent_worry_about_additional_panic_attack'
      'persistent_concern_about_panic_attack_consequences','persistent_worry_about_panic_attack_consequences'},
     {'maladaptive_change_in_behaviour'}, 1], #B
    [{'not_attributable_to_the_physiological_effects_of_a_substance', 
      'not_attributable_to_the_physiological_effects_of_another_medical_condition'}], #C
    [{'not_better_explained_by_social_anxiety_disorder',
      'not_better_explained_by_specific_phobia',
      'not_better_explained_by_obsessive_compulsive_disorder',
      'not_better_explained_by_posttraumatic_stress_disorder',
      'not_better_explained_by_separation_anxiety_disorder'}]  #D
] #3,119,485,608 CSSCs

### Testing

In [7]:
# simple test with output
gen_all_comb(gens3, ret=True)


189 symptom combinations generated!


[{'f1', 'h2', 'i2'},
 {'f1', 'h1', 'i2'},
 {'f1', 'g1', 'i2'},
 {'f1', 'h2', 'i1'},
 {'f1', 'g1', 'h2'},
 {'f1', 'h1', 'i1'},
 {'f1', 'g1', 'i1'},
 {'f1', 'g1', 'h1'},
 {'f1', 'h2', 'i1', 'i2'},
 {'f1', 'h1', 'h2', 'i2'},
 {'f1', 'g1', 'h2', 'i2'},
 {'f1', 'h1', 'i1', 'i2'},
 {'f1', 'g1', 'i1', 'i2'},
 {'f1', 'g1', 'h1', 'i2'},
 {'f1', 'h1', 'h2', 'i1'},
 {'f1', 'g1', 'h2', 'i1'},
 {'f1', 'g1', 'h1', 'h2'},
 {'f1', 'g1', 'h1', 'i1'},
 {'f1', 'h1', 'h2', 'i1', 'i2'},
 {'f1', 'g1', 'h2', 'i1', 'i2'},
 {'f1', 'g1', 'h1', 'h2', 'i2'},
 {'f1', 'g1', 'h1', 'i1', 'i2'},
 {'f1', 'g1', 'h1', 'h2', 'i1'},
 {'f1', 'g1', 'h1', 'h2', 'i1', 'i2'},
 {'f2', 'h2', 'i2'},
 {'f2', 'h1', 'i2'},
 {'f2', 'g1', 'i2'},
 {'f2', 'h2', 'i1'},
 {'f2', 'g1', 'h2'},
 {'f2', 'h1', 'i1'},
 {'f2', 'g1', 'i1'},
 {'f2', 'g1', 'h1'},
 {'f2', 'h2', 'i1', 'i2'},
 {'f2', 'h1', 'h2', 'i2'},
 {'f2', 'g1', 'h2', 'i2'},
 {'f2', 'h1', 'i1', 'i2'},
 {'f2', 'g1', 'i1', 'i2'},
 {'f2', 'g1', 'h1', 'i2'},
 {'f2', 'h1', 'h2', 'i1'},
 

In [8]:
gen_all_comb(gens_mdd_A, ret=False)


7,283,511 symptom combinations generated!


In [9]:
gen_all_comb(gens_persistent_depressive, ret=False)


63,567 symptom combinations generated!
